----
<div style="display: flex; align-items: center;">
    <img src="https://developers.lseg.com/content/dam/devportal/icons/logo/lseg-logo.svg" width="20%" style="vertical-align: top;">
    <h1 style="margin-left: 20px;">Data Library for Python</h1>
</div>

----

## Content layer - Custom Instruments - Manage and Search
This notebook demonstrates how to create, retrieve, update, search for and delete a formula-based custom instrument with `lseg.data.content.custom_instruments`.

Custom instruments are stored on the service under your account. The examples use a unique symbol to avoid overwriting an existing instrument. Run the cleanup cell before closing the session, including if an intermediate request fails. Re-running the creation cell creates another instrument; delete the previous one first.

**Prerequisites:** a configured LSEG Data session, access to Custom Instruments and the underlying market data. For a platform session, creation, modification and deletion require the `trapi.synthetic.crud` scope. Authentication, entitlement and service errors are not suppressed.

**API availability:** the local tests for baskets and user-defined continuous contracts (UDC) are skipped because their creation interfaces were temporarily removed. The public `manage` module exposes `create_formula`, but not `create_basket` or `create_udc`; this series therefore uses formula instruments only.

The other standalone notebooks cover [historical events and summaries](EX-2.12.02-CustomInstruments-HistoricalData.ipynb) and [streaming](EX-2.12.03-CustomInstruments-Streaming.ipynb).

#### Learn more

Visit the [LSEG Developer Portal](https://developers.lseg.com/en) for learning materials about the LSEG Data Library for Python.

#### Getting Help and Support

For questions about the API, visit the [LSEG Developer Community](https://community.developers.lseg.com/).

----

## Set the configuration file location
As in the other Content layer examples, set `LD_LIB_CONFIG_PATH` to the project's `Configuration` directory. Run the notebook with its own directory as the working directory. Edit `lseg-data.config.json` to configure the session and credentials; do not put credentials in the notebook.

In [ ]:
import os
os.environ["LD_LIB_CONFIG_PATH"] = "../../../Configuration"

## Some Imports to start with

In [ ]:
from datetime import date
from uuid import uuid4

import lseg.data as ld
from lseg.data.content import custom_instruments as ci
from lseg.data.errors import LDError

## Open the data session
`open_session()` uses the default session in the configuration file. To select a configured platform session explicitly, use `ld.open_session("platform.rdp")` instead.

In [ ]:
ld.open_session()

## Create a formula instrument
The tests use the formula `GBP=*3`. Here, `GBP=` is the underlying RIC and `*3` scales its value. The `currency` parameter describes the custom instrument; it does not add a currency conversion to the formula.

`instrument_name` is limited to 16 characters. A holiday can be supplied as a `Holiday` object or a dictionary with `date` and `reason`. Use the **full symbol returned by the service**, typically `S)<name>.<owner UUID>`, in subsequent requests rather than constructing the owner suffix yourself.

In [ ]:
requested_symbol = f"Demo{uuid4().hex[:8]}"
holiday_year = date.today().year

instrument = ci.manage.create_formula(
    symbol=requested_symbol,
    formula="GBP=*3",
    instrument_name="GBP Triple",
    exchange_name="8080",
    currency="USD",
    time_zone="LON",
    description="Formula instrument created by the Custom Instruments notebook",
    holidays=[
        ci.manage.Holiday(date=f"{holiday_year}-12-25", name="Christmas Day"),
        {"date": f"{holiday_year}-12-26", "reason": "Boxing Day"},
    ],
)
symbol = instrument.symbol
print(symbol)
instrument.raw

## Retrieve the instrument definition
`manage.get` returns an instrument object, not a pricing response. Its `raw` property exposes the definition returned by the service. Use `events`, `summaries` or a stream to retrieve calculated values.

In [ ]:
retrieved = ci.manage.get(universe=symbol)
print(retrieved.symbol, retrieved.formula, retrieved.currency)
retrieved.raw

## Update and save
Changing an object's properties only updates the local object until `save()` is called. Retrieve the instrument again to inspect the persisted definition.

In [ ]:
retrieved.formula = "GBP=*2"
retrieved.instrument_name = "GBP Double"
retrieved.description = "Updated formula: twice the underlying GBP= value"
retrieved.save()

updated = ci.manage.get(universe=symbol)
updated.raw

## Search instruments owned by the session user
`search.Definition().get_data()` retrieves instrument definitions owned by the authenticated user. `response.data.df` provides a DataFrame and `response.data.raw` provides the service payload.

The example deliberately uses the default search. Tests of the type and limit filters are currently skipped. The final expression filters the returned DataFrame locally to the symbol created above; search results can take time to reflect a new instrument.

In [ ]:
response = ci.search.Definition().get_data()
response.data.df

In [ ]:
owned_instruments = response.data.df
owned_instruments.loc[owned_instruments["symbol"] == symbol]

## Delete the example instrument
**This deletes only the instrument created by this notebook.** Do not replace this with a loop deleting search results, which may include your other instruments. The equivalent object-oriented operation is `updated.delete()`; use one approach, not both. If an earlier cell fails after creation, run this cleanup cell before closing the session.

In [ ]:
ci.manage.delete(universe=symbol)

### Inspect the expected not-found error
Retrieving a deleted instrument raises `LDError` with code 404. Only this expected error is handled; other errors are re-raised.

In [ ]:
try:
    ci.manage.get(universe=symbol)
except LDError as error:
    if error.code != 404:
        raise
    print(f"Expected after deletion: {error}")
else:
    raise RuntimeError(f"The deleted instrument is still retrievable: {symbol}")

## Close the session

In [ ]:
ld.close_session()